# Trening modelu segmentacji Kraken na polskich stronach EHRI

Fine-tuning domyslnego modelu segmentacji Kraken (blla) na 15 polskich stronach EHRI.

**Cel:** poprawic CER z 14,25% (domyslny segmenter) do ~11% (lepsza segmentacja).

**Metoda:**
1. Pobierz 15 polskich stron .tif + ALTO XML z EHRI
2. Podzial: 12 stron train, 3 strony validation
3. Fine-tune domyslnego modelu segmentacji z ketos segtrain --load
4. Ewaluacja: Kraken e2e z custom segmenter vs domyslny

**Uwaga:** 15 stron to malo - uzywamy fine-tuningu (nie from scratch) + augmentacji.

In [ ]:
import subprocess, sys, os
from pathlib import Path
EHRI_DATASET_REPO = 'PiotrSty/ehri-dataset'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
# Checkout latest main (full hash, not short — short hashes fail on Kaggle)
_full = subprocess.run(['git','-C',str(repo),'rev-parse','origin/main'],capture_output=True,text=True)
CODE_REVISION = _full.stdout.strip() or 'main'
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))

# Pin kraken==7.1.1 — API verified against this tag. kraken>=7.0 would float
# to future 7.x/8.x and could silently break. kraken 7.1.1 requires
# huggingface_hub>=0.23 (is_offline_mode) and torch>=2.9 (pip will pull it).
# Removed unused albumentations/opencv-python-headless.
subprocess.run([sys.executable,'-m','pip','install',
    'kraken==7.1.1',
    'jiwer',
    'pillow',
    'huggingface_hub>=0.23',
], check=True)
# Force-reimport in case huggingface_hub was already cached stale
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.'):
        del sys.modules[_mod]
from importlib.metadata import version as _pkg_version
print('IMPORT_OK', 'kraken', _pkg_version('kraken'), 'huggingface_hub', _pkg_version('huggingface_hub'))

In [ ]:
import torch, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
assert torch.cuda.is_available(), "GPU required; select Kaggle GPU T4."
print("GPU:", torch.cuda.get_device_name(0))

ehri_dir = Path("/kaggle/working/ehri-polish")
ehri_dir.mkdir(parents=True, exist_ok=True)
all_files = list_repo_files(EHRI_DATASET_REPO, repo_type="dataset")
polish_files = [f for f in all_files if "polish" in f and (f.endswith(".tif") or f.endswith(".xml"))]
for f in polish_files:
    path = hf_hub_download(EHRI_DATASET_REPO, f, repo_type="dataset")
    shutil.copy(path, ehri_dir / Path(f).name)

kraken_model_path = hf_hub_download(EHRI_DATASET_REPO, "models/polish_nfd_9313.mlmodel", repo_type="dataset")

# Sanity-check: fail fast with a clear message if HF returned no files
tifs = sorted(ehri_dir.glob("*.tif"))
xmls = sorted(ehri_dir.glob("*.xml"))
if len(tifs) == 0:
    raise RuntimeError(f"No .tif files downloaded from {EHRI_DATASET_REPO}. Check repo contents.")
if len(xmls) == 0:
    raise RuntimeError(f"No .xml files downloaded from {EHRI_DATASET_REPO}. Check repo contents.")
# Verify each .tif has a matching .xml (and vice versa)
tif_stems = {p.stem for p in tifs}
xml_stems = {p.stem for p in xmls}
missing_xml = tif_stems - xml_stems
missing_tif = xml_stems - tif_stems
if missing_xml:
    raise RuntimeError(f"TIFFs without matching XML: {sorted(missing_xml)}")
if missing_tif:
    raise RuntimeError(f"XMLs without matching TIFF: {sorted(missing_tif)}")

print("Kraken recognition model:", kraken_model_path)
print("Polish pages (.tif):", len(tifs))
print("ALTO XML:", len(xmls))
assert len(tifs) >= 15, f"Expected >=15 pages, got {len(tifs)}"

In [ ]:
import random
random.seed(42)
pages = sorted(ehri_dir.glob("*.tif"))
assert len(pages) >= 3, f"Need >=3 pages for train/val split, got {len(pages)}"
random.shuffle(pages)
val_pages = pages[:3]
train_pages = pages[3:]
print(f"Train: {len(train_pages)} pages")
for p in train_pages:
    print(f"  {p.name}")
print(f"Validation: {len(val_pages)} pages")
for p in val_pages:
    print(f"  {p.name}")

train_manifest = ehri_dir / "train_manifest.txt"
val_manifest = ehri_dir / "val_manifest.txt"
# Manifests contain XML paths (one per line) — correct for -f xml
# (verified: kraken/ketos/segmentation.py:273-276, kraken/train/blla.py:81-88)
train_manifest.write_text("\n".join(str(p.with_suffix(".xml")) for p in train_pages))
val_manifest.write_text("\n".join(str(p.with_suffix(".xml")) for p in val_pages))
print(f"\nManifests written:")
print(f"  Train: {train_manifest} ({len(train_pages)} entries)")
print(f"  Val: {val_manifest} ({len(val_pages)} entries)")

In [ ]:
# Find the default Kraken segmentation model (blla.mlmodel bundled with the package)
from importlib import resources
import kraken

default_seg = None
# Primary: importlib.resources (correct API for package data)
try:
    cand = resources.files('kraken').joinpath('blla.mlmodel')
    if cand.is_file():
        default_seg = str(cand)
except Exception:
    pass

# Fallback: rglob the package directory
if default_seg is None:
    from pathlib import Path
    kraken_dir = Path(kraken.__file__).parent
    matches = list(kraken_dir.rglob('blla.mlmodel'))
    if matches:
        default_seg = str(matches[0])

print(f'Default seg model: {default_seg}')
if default_seg is None:
    print('WARNING: blla.mlmodel not found — training will be from scratch')

In [ ]:
# Training: fine-tune the default Kraken segmentation model (Kraken 7.1.1)
import subprocess, shutil, glob
from pathlib import Path

output_dir = '/kaggle/working/polish_seg'  # -o is a DIRECTORY (ModelCheckpoint dirpath)
ketos_bin = shutil.which('ketos') or 'ketos'

# Kraken 7.1: -d and --workers are GLOBAL options (before the subcommand).
# segtrain -o is the checkpoint directory; best weights are saved inside it
# as best_<score>.safetensors (kraken/ketos/segmentation.py:374).
cmd = [
    ketos_bin,
    '-d', 'cuda',
    '--workers', '2',
    'segtrain',
    '-f', 'xml',
    '-t', str(train_manifest),
    '-e', str(val_manifest),
    '--augment',
    '-o', output_dir,
    '-N', '50',
    '--weights-format', 'safetensors',
]

if default_seg:
    cmd += ['-i', default_seg, '--resize', 'new']

print('Training command:')
print(' '.join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout[-5000:])
print('STDERR:', result.stderr[-5000:])
print('Return code:', result.returncode)

# segtrain saves best_<score>.safetensors inside the -o directory.
# Old glob (polish_seg*.safetensors) never matched because the '/' separator
# blocked the wildcard and the filename is best_*, not polish_seg*.
trained = sorted(glob.glob(f'{output_dir}/best_*.safetensors')
                 + glob.glob(f'{output_dir}/*.safetensors'))
print('\nTrained model files:', trained)

if result.returncode != 0:
    raise RuntimeError(f'ketos segtrain failed with return code {result.returncode}. See STDERR above.')

In [ ]:
# Evaluation: Kraken e2e with custom segmenter vs default
# Uses kraken.tasks API (SegmentationTaskModel / RecognitionTaskModel) — the
# high-level loaders that accept both .mlmodel and .safetensors paths.
# The old blla.segment()/rpred()/models.load_any are deprecated in 7.1.1 and
# blla.segment(model=<string>) crashes because it does model=[model] then
# iterates over string characters (kraken/blla.py:315-320).
import jiwer, warnings, glob
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from kraken.tasks import SegmentationTaskModel, RecognitionTaskModel
from kraken.configs import SegmentationInferenceConfig, RecognitionInferenceConfig

warnings.filterwarnings('ignore')
ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def load_page_gt_from_alto(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        if text:
            lines.append(text)
    return lines

# Load recognition model (loads both .mlmodel and .safetensors via entry points)
recognizer = RecognitionTaskModel.load_model(kraken_model_path)
rec_config = RecognitionInferenceConfig(device='cuda')

ehri_dir = Path('/kaggle/working/ehri-polish')
val_xmls = [ehri_dir / p.with_suffix('.xml').name for p in val_pages]

# Find trained model: segtrain saves best_<score>.safetensors inside -o dir
output_dir = '/kaggle/working/polish_seg'
trained = sorted(glob.glob(f'{output_dir}/best_*.safetensors')
                 + glob.glob(f'{output_dir}/*.safetensors'))
custom_seg_path = trained[0] if trained else None
print(f'Custom segmenter: {custom_seg_path}')

for seg_name, seg_path in [
    ('default', None),
    ('custom', custom_seg_path),
]:
    if seg_path is None and seg_name == 'custom':
        print(f'\n=== {seg_name}: model not found, skipping ===')
        continue
    print(f'\n=== Kraken e2e + {seg_name} segmenter ===')
    # SegmentationTaskModel.load_model(path) accepts a string path and loads
    # both .mlmodel and .safetensors. load_model(None) loads the default blla.
    seg_model = SegmentationTaskModel.load_model(seg_path)
    seg_config = SegmentationInferenceConfig(device='cuda')
    all_refs, all_hyps = [], []
    for xml_path in val_xmls:
        page_path = xml_path.with_suffix('.tif')
        gt_lines = load_page_gt_from_alto(xml_path)
        img = Image.open(page_path).convert('L')
        segmentation = seg_model.predict(img, seg_config)
        hyp_lines = [record.prediction.strip() for record in recognizer.predict(img, segmentation, rec_config)]
        # Skip pages with zero lines in either GT or OCR for CER/WER
        if not gt_lines or not hyp_lines:
            print(f'  {page_path.name}: {len(gt_lines)} GT, {len(hyp_lines)} OCR — SKIPPED (empty)')
            continue
        ref = '\n'.join(gt_lines)
        hyp = '\n'.join(hyp_lines)
        all_refs.append(ref)
        all_hyps.append(hyp)
        print(f'  {page_path.name}: {len(gt_lines)} GT, {len(hyp_lines)} OCR')
    if all_refs:
        cer = jiwer.cer(all_refs, all_hyps)
        wer = jiwer.wer(all_refs, all_hyps)
        print(f'  CER: {cer:.4f}  ({cer*100:.2f}%)')
        print(f'  WER: {wer:.4f}  ({wer*100:.2f}%)')